# Regularization in Machine Learning: Ridge and Lasso

When training models like Linear Regression, the model can sometimes assign extremely large weights to certain features, overcomplicating the pattern to perfectly fit the training data (a phenomenon known as **overfitting**).
To achieve this "perfect fit," the algorithm might do something crazy, like say:

$Price = (5000 \times Age) - (4999 \times NumberOfWindows)$ \
It artificially inflates the weights to huge values just so they mathematically cancel out perfectly for the training data. The model has essentially memorized the training data (High Variance / Overfitting). \
But the second you give it a new house to predict, those massive weights will cause the prediction to explode and be completely wrong.

**Regularization** is the technique of adding a penalty term to our cost function, which discourages the model from assigning excessively large weights.

In this notebook, we will explore the two most common types of regularization:
1. **Ridge (L2 Regularization)** - Penalizes the *square* of the weights.
2. **Lasso (L1 Regularization)** - Penalizes the *absolute value* of the weights (acting as an automatic feature selector).


## 1. Ridge Regression (L2 Regularization)

In standard Linear Regression, the cost function is the Mean Squared Error (MSE).
In Ridge Regression, we add $\lambda \sum w_i^2$ out to our cost, where $\lambda$ (or alpha) controls how much we penalize the weights.

New Cost = Old Cost (MSE) + $\lambda (w_1^2 + w_2^2 + w_3^2 ...)$

If a weight $w = 10$, the penalty added to the cost is $10^2 = 100$.
The algorithm will forcefully shrink that weight down to an extremely small number (like 0.01) to minimize the Cost function. <br> <br>
**Result:** It forces all the features to have very small, evenly distributed weights. No single feature is allowed to dominate the prediction. This makes the model incredibly stable and robust to new data (prevents overfitting). <br> <br>
**Key Characteristic of Ridge:** It will shrink the weights of useless features to be infinitesimally small (0.0001), but it rarely forces them to be exactly 0. It keeps all features in the model.




Let's build `MyRidgeReg` from scratch using Gradient Descent!


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

class MyRidgeReg:
    def __init__(self, learning_rate=0.01, iterations=1000, alpha=1.0):
        self.lr = learning_rate
        self.iterations = iterations
        self.alpha = alpha  # Regularization strength
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.iterations):
            # 1. Predictions
            y_pred = np.dot(X, self.weights) + self.bias
            
            # 2. Gradients calculate
            # dw includes the L2 penalty derived component: + (alpha/n_samples) * weights
            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y)) + (self.alpha / n_samples) * self.weights
            db = (1 / n_samples) * np.sum(y_pred - y)

            # 3. Update weights
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias


## 2. Lasso Regression (L1 Regularization)

In Lasso Regulation, the cost penalty is $\lambda \sum |w_i|$.
Because the derivative of an absolute value is a constant step (its sign, ignoring $w=0$), Lasso tends to push less important feature weights exactly to **zero**. This is brilliant for feature selection!

Let's build `MyLassoReg` from scratch. We will use the subgradient of the absolute function, meaning we take `sign(weights)`.


In [3]:
class MyLassoReg:
    def __init__(self, learning_rate=0.01, iterations=1000, alpha=1.0):
        self.lr = learning_rate
        self.iterations = iterations
        self.alpha = alpha
        self.weights = None
        self.bias = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0

        for _ in range(self.iterations):
            y_pred = np.dot(X, self.weights) + self.bias
            
            # dw includes L1 penalty component: + (alpha/n_samples) * sign(weights)
            dw = (1 / n_samples) * np.dot(X.T, (y_pred - y)) + (self.alpha / n_samples) * np.sign(self.weights)
            db = (1 / n_samples) * np.sum(y_pred - y)

            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias


## 3. Testing and Comparing Models

Let's create a synthetic dataset that has some "useless" features to observe how Ridge and Lasso handle them.
We'll compare our custom implementations with zero regularization (standard Linear Regression) as well.


In [4]:
from sklearn.preprocessing import StandardScaler

# 1. Create a dataset
# We create 10 features, but only 2 of them actually contain useful information
np.random.seed(42)
X = np.random.randn(500, 10)
# y only really depends on first two features
y = 3 * X[:, 0] + 5 * X[:, 1] + np.random.randn(500) * 2 

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# --- Train Models ---
# Unregularized (Alpha = 0)
no_reg = MyRidgeReg(alpha=0)
no_reg.fit(X_train, y_train)

# Ridge
ridge = MyRidgeReg(alpha=50.0)
ridge.fit(X_train, y_train)

# Lasso
lasso = MyLassoReg(alpha=50.0)
lasso.fit(X_train, y_train)

# --- Evaluate ---
print("--- MSE Performance ---")
print(f"No Regularization MSE: {mean_squared_error(y_test, no_reg.predict(X_test)):.3f}")
print(f"Ridge MSE: {mean_squared_error(y_test, ridge.predict(X_test)):.3f}")
print(f"Lasso MSE: {mean_squared_error(y_test, lasso.predict(X_test)):.3f}")

print("\n--- Learned Weights (First 5 vs Last 5) ---")
print("Notice how Lasso squashes useless feature weights exactly to 0!")
print("No Reg Weights: \n", np.round(no_reg.weights, 3))
print("Ridge Weights: \n", np.round(ridge.weights, 3))
print("Lasso Weights: \n", np.round(lasso.weights, 3))


--- MSE Performance ---
No Regularization MSE: 3.929
Ridge MSE: 4.136
Lasso MSE: 3.806

--- Learned Weights (First 5 vs Last 5) ---
Notice how Lasso squashes useless feature weights exactly to 0!
No Reg Weights: 
 [ 2.822  5.353 -0.013 -0.016  0.08  -0.078 -0.     0.111  0.255 -0.034]
Ridge Weights: 
 [ 2.554e+00  4.775e+00 -4.000e-03 -3.100e-02  1.210e-01 -7.600e-02
 -2.000e-02  1.060e-01  2.450e-01 -7.300e-02]
Lasso Weights: 
 [ 2.708e+00  5.257e+00 -1.000e-03 -1.000e-03  1.000e-03 -0.000e+00
 -1.000e-03  2.000e-03  1.320e-01 -1.000e-03]


## 4. Comparing with scikit-learn

In industry, you will typically use `sklearn.linear_model.Ridge` and `sklearn.linear_model.Lasso`. Let's verify our findings with the official library.


In [5]:
from sklearn.linear_model import Ridge, Lasso, LinearRegression

sk_no_reg = LinearRegression()
sk_no_reg.fit(X_train, y_train)

# NOTE: sklearn implementation handles alpha slightly differently scale-wise, 
# so we might use a larger/smaller alpha to see similar effects.
sk_ridge = Ridge(alpha=50.0)
sk_ridge.fit(X_train, y_train)

# sk_lasso uses coordinate descent
sk_lasso = Lasso(alpha=0.5) 
sk_lasso.fit(X_train, y_train)

print("SKLearn Linear Reg Weights: \n", np.round(sk_no_reg.coef_, 3))
print("SKLearn Ridge Weights: \n", np.round(sk_ridge.coef_, 3))
print("SKLearn Lasso Weights: \n", np.round(sk_lasso.coef_, 3))


SKLearn Linear Reg Weights: 
 [ 2.822  5.354 -0.013 -0.016  0.079 -0.078 -0.     0.111  0.254 -0.034]
SKLearn Ridge Weights: 
 [ 2.554e+00  4.775e+00 -3.000e-03 -3.100e-02  1.210e-01 -7.600e-02
 -2.000e-02  1.060e-01  2.450e-01 -7.300e-02]
SKLearn Lasso Weights: 
 [ 2.365  4.92   0.    -0.     0.    -0.    -0.     0.     0.    -0.   ]
